In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/dog-breed-identification.zip" -d "/content/dataset_folder"

In [ ]:
import numpy as np
from sklearn.metrics import log_loss
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset, Features, Image, ClassLabel, Value
from torchvision.transforms import Compose, RandomResizedCrop, RandomHorizontalFlip, Resize, CenterCrop, ToTensor, Normalize
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
import torch
import os
import gc

In [ ]:
base_dir = "/content/dataset_folder"

train_df = pd.read_csv(f"{base_dir}/labels.csv")
train_df = train_df.rename(columns={'id': 'file_name', 'breed': 'label'})
train_df['file_name'] = train_df['file_name'].apply(lambda x: f"{x}.jpg")
train_df.to_csv(f"{base_dir}/train/metadata.csv", index=False)

unique_labels = sorted(train_df['label'].unique().tolist())

features_train = Features({
    'file_name': Value('string'),
    'label': ClassLabel(names=unique_labels),
    'image': Image()
})

dataset = load_dataset(
    "imagefolder",
    data_dir=f"{base_dir}/train",
    features=features_train
)

dataset = dataset["train"].train_test_split(test_size=0.15, seed=42)
dataset["validation"] = dataset.pop("test")

test_dataset = load_dataset("imagefolder", data_dir=f"{base_dir}/test", drop_labels=True, split="train")

save_path = "/content/drive/MyDrive/dog_breed_project"

Resolving data files:   0%|          | 0/10223 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/10357 [00:00<?, ?it/s]

In [ ]:
class_counts = train_df['label'].value_counts()

print(f"Всего пород: {len(class_counts)}")
print(f"Максимум фото на породу: {class_counts.max()} ({class_counts.idxmax()})")
print(f"Минимум фото на породу: {class_counts.min()} ({class_counts.idxmin()})")
print(f"Среднее количество фото: {class_counts.mean():.2f}")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Коэффициент дисбаланса: {imbalance_ratio:.2f}")

Всего пород: 120
Максимум фото на породу: 126 (scottish_deerhound)
Минимум фото на породу: 66 (eskimo_dog)
Среднее количество фото: 85.18
Коэффициент дисбаланса: 1.91


In [ ]:
import evaluate
import numpy as np

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])

    if "label" in examples[0]:
        labels = torch.tensor([int(example["label"]) for example in examples], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}

    return {"pixel_values": pixel_values}

def test_collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    return {"pixel_values": pixel_values}

from sklearn.metrics import log_loss
import torch.nn.functional as F

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()
    loss = log_loss(labels, probs, labels=list(range(len(unique_labels))))
    predictions = np.argmax(logits, axis=1)
    acc = (predictions == labels).mean()

    return {
        "log_loss": loss,
        "accuracy": acc
    }

In [ ]:
def train_predict_save(model_name, model_path):
    labels = dataset["train"].features["label"].names
    label2id, id2label = dict(), dict()
    for i, label in enumerate(labels):
        label2id[label] = str(i)
        id2label[str(i)] = label

    test_dataset.reset_format()
    test_ids = []
    for i in range(len(test_dataset)):
        img_obj = test_dataset[i]["image"]
        path = getattr(img_obj, 'filename', f"img_{i}.jpg")
        file_id = os.path.splitext(os.path.basename(path))[0]
        test_ids.append(file_id)

    all_metrics = {}

    image_processor = AutoImageProcessor.from_pretrained(model_path)
    size = image_processor.size.get("shortest_edge", image_processor.size.get("height", 224))

    normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

    train_transforms = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize])
    val_transforms = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

    dataset["train"].set_transform(lambda b: {"pixel_values": [train_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    dataset["validation"].set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    test_dataset.set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]]})

    model = AutoModelForImageClassification.from_pretrained(
        model_path,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True
    )

    current_output_dir = os.path.join(save_path, model_name)
    training_args = TrainingArguments(
        output_dir=current_output_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        fp16=True,
        logging_steps=50,
        metric_for_best_model="loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        processing_class=image_processor
    )

    trainer.train()
    trainer.save_model(f"{save_path}/final_best_model")

    print(f"Создание сабмишена для {model_name}...")
    test_results = trainer.predict(test_dataset)
    logits = torch.from_numpy(test_results.predictions)
    probabilities = F.softmax(logits, dim=-1).numpy()

    sub_df = pd.DataFrame(probabilities, columns=unique_labels)
    sub_df = sub_df.reindex(sorted(sub_df.columns), axis=1)
    sub_df.insert(0, 'id', test_ids)
    sub_df.to_csv(f"submission_{model_name}.csv", index=False)

In [ ]:
train_predict_save("vit-base", "google/vit-base-patch16-224")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([120, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([120])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.262302,0.872308,0.838331
2,0.805341,0.551882,0.874185
3,0.603085,0.488837,0.887223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для vit-base...


In [ ]:
import evaluate
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

def train_predict_save_accuracy(model_name, model_path, num_epoch):
    labels = dataset["train"].features["label"].names
    label2id, id2label = dict(), dict()
    for i, label in enumerate(labels):
        label2id[label] = str(i)
        id2label[str(i)] = label

    test_dataset.reset_format()
    test_ids = []
    for i in range(len(test_dataset)):
        img_obj = test_dataset[i]["image"]
        path = getattr(img_obj, 'filename', f"img_{i}.jpg")
        file_id = os.path.splitext(os.path.basename(path))[0]
        test_ids.append(file_id)

    all_metrics = {}

    image_processor = AutoImageProcessor.from_pretrained(model_path)
    size = image_processor.size.get("shortest_edge", image_processor.size.get("height", 224))

    normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

    train_transforms = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize])
    val_transforms = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

    dataset["train"].set_transform(lambda b: {"pixel_values": [train_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    dataset["validation"].set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    test_dataset.set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]]})

    model = AutoModelForImageClassification.from_pretrained(
        model_path,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True
    )

    current_output_dir = os.path.join(save_path, model_name)
    training_args = TrainingArguments(
        output_dir=current_output_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=num_epoch,
        fp16=True,
        logging_steps=50,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        processing_class=image_processor
    )

    trainer.train()
    trainer.save_model(f"{save_path}/final_best_model-{model_name}")

    print(f"Создание сабмишена для {model_name}...")
    test_results = trainer.predict(test_dataset)
    logits = torch.from_numpy(test_results.predictions)
    probabilities = F.softmax(logits, dim=-1).numpy()

    sub_df = pd.DataFrame(probabilities, columns=unique_labels)
    sub_df = sub_df.reindex(sorted(sub_df.columns), axis=1)
    sub_df.insert(0, 'id', test_ids)
    sub_df.to_csv(f"submission_{model_name}.csv", index=False)

In [ ]:
import torch
import gc
import os
import torch.nn.functional as F

models_to_test = [
    {"id": "vit_base-224", "path": "google/vit-base-patch16-224"},
    {"id": "swin_tiny", "path": "microsoft/swin-tiny-patch4-window7-224"},
    {"id": "convnext_tiny", "path": "facebook/convnextv2-tiny-1k-224"}
]

labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

test_dataset.reset_format()
test_ids = []
for i in range(len(test_dataset)):
    img_obj = test_dataset[i]["image"]
    path = getattr(img_obj, 'filename', f"img_{i}.jpg")
    file_id = os.path.splitext(os.path.basename(path))[0]
    test_ids.append(file_id)

all_metrics = {}

for experiment in models_to_test:
    model_name = experiment["id"]
    model_path = experiment["path"]

    print(f"\n" + "="*50)
    print(f"ЗАПУСК ЭКСПЕРИМЕНТА: {model_name}")
    print(f"Путь: {model_path}")
    print("="*50)

    image_processor = AutoImageProcessor.from_pretrained(model_path)
    size = image_processor.size.get("shortest_edge", image_processor.size.get("height", 224))

    normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

    train_transforms = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize])
    val_transforms = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

    dataset["train"].set_transform(lambda b: {"pixel_values": [train_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    dataset["validation"].set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
    test_dataset.set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]]})

    model = AutoModelForImageClassification.from_pretrained(
        model_path,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True
    )

    current_output_dir = os.path.join(save_path, model_name)
    training_args = TrainingArguments(
        output_dir=current_output_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        fp16=True,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
        seed=42,
        data_seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        processing_class=image_processor
    )

    trainer.train()

    eval_results = trainer.evaluate()
    all_metrics[model_name] = eval_results["eval_accuracy"]

    print(f"Создание сабмишена для {model_name}...")

    test_results = trainer.predict(test_dataset)
    logits = torch.from_numpy(test_results.predictions)
    probabilities = F.softmax(logits, dim=-1).numpy()

    sub_df = pd.DataFrame(probabilities, columns=unique_labels)
    sub_df = sub_df.reindex(sorted(sub_df.columns), axis=1)
    sub_df.insert(0, 'id', test_ids)
    sub_df.to_csv(f"submission_{model_name}.csv", index=False)

    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ:")
for m_id, acc in all_metrics.items():
    print(f"Модель: {m_id:15} | Accuracy: {acc:.4f}")
print("="*50)


ЗАПУСК ЭКСПЕРИМЕНТА: vit_base-224
Путь: google/vit-base-patch16-224


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([120])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([120, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Epoch,Training Loss,Validation Loss,Accuracy
1,1.968989,1.436156,0.852673
2,1.140343,0.763283,0.884615
3,0.894271,0.638179,0.891134


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для convnext_tiny...

РЕЗУЛЬТАТЫ:
Модель: vit_base-224    | Accuracy: 0.8885
Модель: swin_tiny       | Accuracy: 0.8579
Модель: convnext_tiny   | Accuracy: 0.8911


convnext без AutoAugment

In [ ]:
model_path = "facebook/convnextv2-base-22k-384"
model_name = "convnext-384"

train_predict_save_accuracy(model_name, model_path, 3)

preprocessor_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

ConvNextV2ForImageClassification LOAD REPORT from: facebook/convnextv2-base-22k-384
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([120])            
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 1024]) vs model:torch.Size([120, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.890336,0.511020,0.867666
2,0.699195,0.347606,0.903520
3,0.472097,0.304249,0.907432


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для convnext-384...


Convnext с AutoAugment

In [ ]:
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
model_path = "facebook/convnextv2-base-22k-384"
model_name = "convnext-384-auto-augment"
current_output_dir = os.path.join(save_path, model_name)

image_processor = AutoImageProcessor.from_pretrained(model_path)
size = image_processor.size.get("shortest_edge", image_processor.size.get("height", 224))

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

train_transforms = Compose([
    Resize((size, size)),
    AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
    ToTensor(),
    normalize,
])
val_transforms = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

dataset["train"].set_transform(lambda b: {"pixel_values": [train_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
dataset["validation"].set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]], "label": b["label"]})
test_dataset.set_transform(lambda b: {"pixel_values": [val_transforms(img.convert("RGB")) for img in b["image"]]})

model = AutoModelForImageClassification.from_pretrained(
    model_path,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

training_args = TrainingArguments(
    output_dir=save_path,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=image_processor
)

trainer.train()

print(f"Создание сабмишена для {model_name}...")
test_results = trainer.predict(test_dataset)
logits = torch.from_numpy(test_results.predictions)
probabilities = F.softmax(logits, dim=-1).numpy()

sub_df = pd.DataFrame(probabilities, columns=unique_labels)
sub_df = sub_df.reindex(sorted(sub_df.columns), axis=1)
sub_df.insert(0, 'id', test_ids)
sub_df.to_csv(f"submission_{model_name}.csv", index=False)

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

ConvNextV2ForImageClassification LOAD REPORT from: facebook/convnextv2-base-22k-384
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([120])            
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 1024]) vs model:torch.Size([120, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.469126,0.461501,0.877445
2,0.252711,0.367035,0.895698
3,0.144742,0.317810,0.908083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для convnext-384-auto-augment...


In [ ]:
model_path = "microsoft/swin-base-patch4-window12-384-in22k"
model_name = "swin-22k-384"

train_predict_save_accuracy(model_name, model_path, 5)

Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-base-patch4-window12-384-in22k
Key               | Status   |                                                                                             
------------------+----------+---------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841, 1024]) vs model:torch.Size([120, 1024])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([21841]) vs model:torch.Size([120])            

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.138475,0.718542,0.769231
2,0.947123,0.509349,0.840287
3,0.672622,0.480336,0.846154
4,0.547773,0.467177,0.847458
5,0.442382,0.431162,0.868318


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для swin-22k-384...


In [ ]:
model_path = "facebook/convnextv2-base-22k-384"
model_name = "convnext-384"

train_predict_save_accuracy(model_name, model_path, 5)

The image processor of type `ConvNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

ConvNextV2ForImageClassification LOAD REPORT from: facebook/convnextv2-base-22k-384
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 1024]) vs model:torch.Size([120, 1024])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([120])            

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.897764,0.516580,0.861147
2,0.711851,0.347122,0.909387
3,0.486915,0.359774,0.893742
4,0.485428,0.313164,0.915906
5,0.360353,0.309825,0.908083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Создание сабмишена для convnext-384...


In [ ]:
subs = {
    "swin": pd.read_csv("submission_swin-22k-384.csv"),
    "conv": pd.read_csv("submission_convnext-384.csv"),
}

def make_blend(name, models_with_weights):
    first_key = list(models_with_weights.keys())[0]
    result = subs[first_key][['id']].copy()

    final_probs = np.zeros((len(result), 120))

    for key, weight in models_with_weights.items():
        final_probs += subs[key].iloc[:, 1:].values * weight

    probs_df = pd.DataFrame(final_probs, columns=subs[first_key].columns[1:])
    final_sub = pd.concat([result, probs_df], axis=1)
    final_sub.to_csv(f"ensemble_{name}.csv", index=False)


make_blend("swin_conv_blend", {"swin": 0.6, "conv": 0.4})
make_blend("blend_conv_swin", {"swin": 0.4, "conv": 0.6})
make_blend("all_basic_mean", {"swin": 0.5, "conv": 0.5})